# Fraud detection: Gemma 3 1B + GAT

In [ ]:
# Install once in the selected notebook environment if needed:
# %pip install -r requirements.txt

from pathlib import Path
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GATConv
from sklearn.metrics import f1_score, roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
DATASET_PATH = Path("datasets/reddit.pt")  # change to datasets/instagram.pt when needed
OUTPUT_DIR = Path("artifacts/notebook-reddit")
MODEL_NAME = "google/gemma-3-1b-it"
HIDDEN_DIM, HEADS = 128, 4
NEIGHBORS_PER_NODE = 10
MAX_INPUT_TOKENS, MAX_NEW_TOKENS = 2048, 64
EPOCHS, GAT_LR, LLM_LR = 3, 1e-3, 1e-4
ALPHA, BETA = 0.1, 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DEVICE, DATASET_PATH)

### Data loading and split handling

In [ ]:

try:
    graph = torch.load(DATASET_PATH, map_location="cpu", weights_only=False)
except TypeError:
    graph = torch.load(DATASET_PATH, map_location="cpu")
if not hasattr(graph, "raw_texts") or not hasattr(graph, "edge_index") or not hasattr(graph, "y"):
    raise ValueError("Dataset must contain raw_texts, edge_index, and y")
graph.raw_texts = [str(text) for text in graph.raw_texts]
graph.edge_index = graph.edge_index.long().cpu()
graph.y = graph.y.long().view(-1).cpu()
def split_indices(name):
    mask = getattr(graph, name + "_mask", None)
    if mask is None:
        raise ValueError(f"Dataset has no {name}_mask")
    return mask.nonzero(as_tuple=False).view(-1).long()
train_idx, val_idx = split_indices("train"), split_indices("val")
print(graph, "train nodes:", len(train_idx), "validation nodes:", len(val_idx))

In [ ]:
class GemmaEnhancer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
        self.model = get_peft_model(base, LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], bias="none")).to(DEVICE)
    def prompt(self, text, causal):
        focus = "discriminative features related to the fraud label" if causal else "generic background information unrelated to the fraud label"
        return f"Extract one concise sentence containing {focus}. Do not predict the label. Text: {text[:4000]}\nAnswer:"
    @torch.no_grad()
    def generate(self, texts):
        result = []
        for text in texts:
            answers = []
            for causal in (True, False):
                inputs = self.tokenizer(self.prompt(text, causal), return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(DEVICE)
                tokens = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=self.tokenizer.pad_token_id)
                answer = self.tokenizer.decode(tokens[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().splitlines()
                answers.append(answer[0] if answer else text[:200])
            result.append(answers)
        return result
    def encode(self, texts):
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS).to(DEVICE)
        hidden = self.model.base_model.model(**inputs, output_hidden_states=True, return_dict=True).last_hidden_state
        mask = inputs.attention_mask.unsqueeze(-1)
        return (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
enhancer = GemmaEnhancer()
enhancer.model.print_trainable_parameters()

In [ ]:
class GATClassifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.first = GATConv(in_dim, HIDDEN_DIM, heads=HEADS, concat=True, dropout=0.5)
        self.second = GATConv(HIDDEN_DIM * HEADS, 2, heads=1, concat=False, dropout=0.5)
    def forward(self, x, edge_index):
        x = F.dropout(x, 0.5, self.training)
        x = self.first(x, edge_index).elu()
        return self.second(F.dropout(x, 0.5, self.training), edge_index)
def g_loss(causal_logits, residual_logits, labels):
    disc = F.cross_entropy(causal_logits, labels)
    uniform = torch.full_like(residual_logits, 0.5)
    residual = F.kl_div(F.log_softmax(residual_logits, -1), uniform, reduction="batchmean")
    orthogonal = F.cosine_similarity(causal_logits.flatten(), residual_logits.flatten(), dim=0)
    return disc + ALPHA * residual + BETA * orthogonal
def induced_edges(indices):
    nodes = indices.cpu(); mapping = torch.full((graph.num_nodes,), -1, dtype=torch.long); mapping[nodes] = torch.arange(len(nodes))
    edges = graph.edge_index; keep = (mapping[edges[0]] >= 0) & (mapping[edges[1]] >= 0)
    return mapping[edges[:, keep]].to(DEVICE)
@torch.no_grad()
def semantic_filter(edges, embeddings):
    normalized = F.normalize(embeddings.float(), dim=-1)
    scores = (normalized[edges[0]] * normalized[edges[1]]).sum(-1)
    keep = torch.zeros(edges.shape[1], dtype=torch.bool, device=edges.device)
    for source in edges[0].unique():
        candidates = (edges[0] == source).nonzero().view(-1)
        chosen = torch.topk(scores[candidates], min(NEIGHBORS_PER_NODE, len(candidates))).indices
        keep[candidates[chosen]] = True
    return edges[:, keep]

# Independent alternating training and validation

In [ ]:
with torch.no_grad():
    embedding_dim = enhancer.encode([graph.raw_texts[0]]).shape[-1]
gat = GATClassifier(embedding_dim).to(DEVICE)
llm_optimizer = torch.optim.AdamW(enhancer.model.parameters(), lr=LLM_LR)
gat_optimizer = torch.optim.AdamW(gat.parameters(), lr=GAT_LR, weight_decay=5e-4)
def run_split(indices, training):
    pairs = enhancer.generate([graph.raw_texts[int(i)] for i in indices])
    causal = enhancer.encode([pair[0] for pair in pairs])
    residual = enhancer.encode([pair[1] for pair in pairs])
    with torch.no_grad():
        original = enhancer.encode([graph.raw_texts[int(i)] for i in indices])
    edges = semantic_filter(induced_edges(indices), original)
    labels = graph.y[indices].to(DEVICE)
    gat.train(training); enhancer.model.train(training)
    if training: llm_optimizer.zero_grad(); gat_optimizer.zero_grad()
    with torch.set_grad_enabled(training):
        loss = g_loss(gat(causal, edges), gat(residual, edges), labels)
        if training: loss.backward(); llm_optimizer.step(); gat_optimizer.step()
    with torch.no_grad():
        logits = gat(causal, edges); prediction = logits.argmax(-1).cpu().numpy(); probability = logits.softmax(-1)[:, 1].cpu().numpy()
    metrics = {"loss": float(loss.detach().cpu()), "f1": f1_score(labels.cpu(), prediction, average="macro")}
    metrics["auc"] = roc_auc_score(labels.cpu(), probability) if len(set(labels.cpu().tolist())) > 1 else float("nan")
    return metrics
best = -float("inf")
for epoch in range(EPOCHS):
    train_metrics = run_split(train_idx, True)
    val_metrics = run_split(val_idx, False)
    score = val_metrics["f1"] + val_metrics["auc"]
    print(f"epoch {epoch + 1}: train={train_metrics} val={val_metrics}")
    if score > best:
        best = score; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        torch.save(gat.state_dict(), OUTPUT_DIR / "gat.pt")
        enhancer.model.save_pretrained(OUTPUT_DIR / "gemma-lora")
print("saved to", OUTPUT_DIR)